# Tokenization — companion notebook

Train byte-pair encoding on your own text and watch token counts move with content type. This is the real BPE algorithm, not a simplification.

## 1. Train BPE

Merge the most frequent adjacent pair, repeat.

In [ ]:
from collections import Counter

CORPUS = ("the reader reads the readable reader " * 3 +
          "a leader leads the leading leader " * 3)

def word_freqs(text):
    f = Counter()
    for w in text.split(): f[tuple(w) + ("</w>",)] += 1
    return dict(f)

def best_pair(freqs):
    p = Counter()
    for sym, n in freqs.items():
        for a, b in zip(sym, sym[1:]): p[(a, b)] += n
    return p.most_common(1)[0][0] if p else None

def apply_merge(freqs, pair):
    a, b = pair; out = {}
    for sym, n in freqs.items():
        m, i = [], 0
        while i < len(sym):
            if i < len(sym)-1 and sym[i] == a and sym[i+1] == b:
                m.append(a+b); i += 2
            else:
                m.append(sym[i]); i += 1
        out[tuple(m)] = n
    return out

def train(text, n_merges):
    f, merges = word_freqs(text), []
    for _ in range(n_merges):
        p = best_pair(f)
        if not p: break
        merges.append(p); f = apply_merge(f, p)
    return merges

def tokenize(word, merges):
    s = tuple(word) + ("</w>",)
    for a, b in merges: s = tuple(apply_merge({s: 1}, (a, b)).keys())[0]
    return list(s)

merges = train(CORPUS, 12)
for i, (a, b) in enumerate(merges, 1): print(f"{i:>2}. {a!r}+{b!r} -> {a+b!r}")

## 2. Vocabulary size changes everything

More merges means more single-token words — and a bigger vocabulary to store. This is the compression/coverage trade every tokenizer makes.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WORDS = ["reader", "leader", "readable", "readership", "xylophone"]
sizes = list(range(0, 41, 2))
series = {w: [] for w in WORDS}
for n in sizes:
    m = train(CORPUS, n)
    for w in WORDS: series[w].append(len(tokenize(w, m)))

fig, ax = plt.subplots(figsize=(8, 4))
for w, ys in series.items(): ax.plot(sizes, ys, marker="o", ms=3, label=w)
ax.set_xlabel("merges learned (vocabulary size)")
ax.set_ylabel("tokens for the word")
ax.set_title("Frequent words compress; unseen ones never do")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("'xylophone' shares nothing with the corpus, so no number of merges helps.")
print("That is the shape of a non-English word hitting an English tokenizer.")

## 3. Cost by content type

The ratio that breaks budgets. Compare prose against identifiers.

In [ ]:
SAMPLES = {
    "prose (in corpus)":  "the reader reads the leader",
    "prose (unseen)":     "the xylophone player arrived",
    "identifiers":        "ERR-4021 ORD-99381 SKU-77123",
    "numbers":            "1234567 8901234 5678901",
}
print(f"{'content':<22} {'words':>6} {'tokens':>7} {'tok/word':>9}")
for label, text in SAMPLES.items():
    nw = len(text.split())
    nt = sum(len(tokenize(w, merges)) for w in text.split())
    print(f"{label:<22} {nw:>6} {nt:>7} {nt/nw:>9.1f}")
print("\nA budget built on the first row will be badly wrong for the last two.")

## 4. Your turn

1. Replace `CORPUS` with a sample of your own production text and retrain.
2. Add rows to §3 for your real content — customer messages, code, other languages — and compute tokens-per-word for each.
3. Then do it again with your provider's *real* tokenizer, and compare. The shape will hold; the numbers will not.